In [1]:
import importlib
import data_prep
import features
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

import matplotlib.pyplot as plt

# Prepare Dataset

In [2]:
df = data_prep.prepare_dataset()

In [3]:
df.head(3)

,final_price_per_sqm,area,main_area,net_area,parking_area,total_floors,floor_level,building_year,building_age,transaction_year,...,district_北投區,district_南港區,district_士林區,district_大同區,district_大安區,district_文山區,district_松山區,district_萬華區,building_type_公寓(5樓含以下無電梯),building_type_華廈(10層含以下有電梯)
0,129467,19.32,14.01,19.32,0.0,7,2.0,1980.0,46.0,2026,...,False,False,False,False,False,False,False,False,False,True
1,272458,30.39,16.91,30.39,0.0,10,4.0,2008.0,18.0,2026,...,False,False,False,False,False,True,False,False,False,True
2,76594,91.13,81.23,91.13,0.0,7,-1.0,1978.0,48.0,2026,...,False,False,False,False,False,False,False,False,False,True


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 171 entries, 0 to 176
Data columns (total 28 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   final_price_per_sqm          171 non-null    int64  
 1   area                         171 non-null    float64
 2   main_area                    171 non-null    float64
 3   net_area                     171 non-null    float64
 4   parking_area                 171 non-null    float64
 5   total_floors                 171 non-null    int64  
 6   floor_level                  171 non-null    float64
 7   building_year                171 non-null    float64
 8   building_age                 171 non-null    float64
 9   transaction_year             171 non-null    int32  
 10  transaction_month            171 non-null    int32  
 11  area_ratio                   171 non-null    float64
 12  time_index                   171 non-null    int32  
 13  log_area                 

# Create reusable evaluation function

In [5]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)

    y_pred_test = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    mae = mean_absolute_error(y_test, y_pred_test)

    return rmse, mae

# Prepare Dataset

In [6]:
X_baseline_features = df.drop(columns=['final_price_per_sqm'])
y_baseline_features = df['final_price_per_sqm']

In [7]:
# Drop correlated area features
cols_to_drop = [
    'area',
    'main_area',
    'net_area'
]
X_baseline_features = X_baseline_features.drop(columns=cols_to_drop)

# Simplify time features
X_baseline_features = X_baseline_features.drop(columns=['transaction_year', 'transaction_month', 'building_year'])

# Handle weak/noisy features
X_baseline_features = X_baseline_features.drop(columns=['floor_level'])

In [8]:
X_train_baseline, X_test_baseline, y_train_baseline, y_test_baseline = train_test_split(
    X_baseline_features, y_baseline_features,
    test_size=0.2,
    random_state=42
)

# Prepare Baseline model
### Using clean baseline v2

In [9]:
X_train_baseline_scaled = X_train_baseline.copy()
X_test_baseline_scaled = X_test_baseline.copy()

In [10]:
num_cols = [
    'log_area',
    'area_ratio',
    'building_age',
    'total_floors',
    'parking_area',
    'time_index'
]

scaler = StandardScaler()

X_train_baseline_scaled[num_cols] = scaler.fit_transform(X_train_baseline_scaled[num_cols])
X_test_baseline_scaled[num_cols] = scaler.transform(X_test_baseline_scaled[num_cols])

In [11]:
X_train_baseline_scaled[num_cols].describe()

,log_area,area_ratio,building_age,total_floors,parking_area,time_index
count,1.360000e+02,1.360000e+02,1.360000e+02,1.360000e+02,1.360000e+02,1.360000e+02
mean,4.114356e-16,-2.351061e-16,5.224579e-17,1.044916e-16,-2.612289e-17,-2.677597e-16
std,1.003697e+00,1.003697e+00,1.003697e+00,1.003697e+00,1.003697e+00,1.003697e+00
min,-3.886298e+00,-2.058389e+00,-1.921670e+00,-1.406900e+00,-3.449193e-01,-7.057848e+00
25%,-6.525744e-01,-7.842323e-01,-8.609600e-01,-9.321146e-01,-3.449193e-01,-1.613486e-01
50%,8.223241e-02,3.022323e-02,2.881421e-01,-2.199372e-01,-3.449193e-01,-1.613486e-01
75%,5.765503e-01,8.004184e-01,8.479611e-01,7.296328e-01,-3.449193e-01,4.656059e-01
max,2.228052e+00,1.614044e+00,1.496173e+00,4.053128e+00,7.307108e+00,1.092560e+00


In [12]:
model_baseline = LinearRegression()
rmse_baseline, mae_baseline = evaluate_model(
    model_baseline, 
    X_train_baseline_scaled, X_test_baseline_scaled,
    y_train_baseline, y_test_baseline
)


In [13]:
print(f'Baseline RMSE: {rmse_baseline:,.2f}')
print(f'Baseline MAE: {mae_baseline:,.2f}')

Baseline RMSE: 59,696.36
Baseline MAE: 45,738.04


In [14]:
model_baseline.intercept_

np.float64(275223.6656637115)

In [15]:
feature_importance_baseline = pd.Series(model_baseline.coef_, index = X_baseline_features.columns)
feature_importance_baseline.sort_values(ascending=False)

district_大安區                   106065.067303
district_信義區                    32943.946708
district_中正區                    22003.209481
area_ratio                       6563.704873
parking_area                     6337.311954
district_松山區                    -1806.863153
total_floors                    -5944.897442
has_parking                     -6980.245723
time_index                     -10241.218437
district_大同區                   -14434.816162
log_area                       -20457.293551
building_type_公寓(5樓含以下無電梯)     -25325.380349
district_內湖區                   -25527.250652
district_士林區                   -28502.525593
building_type_華廈(10層含以下有電梯)    -42684.244840
building_age                   -55616.344617
district_文山區                   -64518.390945
district_萬華區                   -67258.170097
district_南港區                   -68250.591653
district_北投區                   -81949.533272
dtype: float64

In [16]:
results = []

results.append({
    "model": "Linear Regression (v2)",
    "rmse": rmse_baseline,
    "mae": mae_baseline,
    "notes": "baseline"
})

# Regularized Models

In [17]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, Lasso

In [18]:
ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0))
])

lasso_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Lasso(alpha=0.01))
])

In [19]:
for name, model in [
    ('Ridge', ridge_pipeline),
    ('Lasso', lasso_pipeline)
]:
    rmse, mae = evaluate_model(model, X_train_baseline, X_test_baseline, y_train_baseline, y_test_baseline)

    results.append({
        'model': name,
        'rmse': rmse,
        'mae': mae,
        'notes': 'regularized linear'
    })

# Tree Based Models

In [20]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

In [21]:
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    random_state=42
)

xgb = XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

In [22]:
for name, model in [
    ('Random Forest', rf),
    ('XGBoost', xgb)
]:
    rmse, mae = evaluate_model(model, X_train_baseline, X_test_baseline, y_train_baseline, y_test_baseline)

    results.append({
        'model': name,
        'rmse': rmse,
        'mae': mae,
        'notes': 'tree-based'
    })

# Hyperparameter Tuning for XGBoost

In [23]:
configs = [
    {'n_estimators':100, 'max_depth': 4, 'learning_rate':0.1},
    {'n_estimators':200, 'max_depth': 6, 'learning_rate':0.05},
    {'n_estimators':300, 'max_depth': 8, 'learning_rate':0.03}
]

In [24]:
for config in configs:
    model = XGBRegressor(**config, random_state=42)

    rmse, mae = evaluate_model(model, X_train_baseline, X_test_baseline, y_train_baseline, y_test_baseline)

    results.append({
        'model': f'XGBoost {config}',
        'rmse': rmse,
        'mae': mae,
        'notes': 'tuned'
    })

# Comparison Table

In [26]:
X_baseline_features.columns

Index(['parking_area', 'total_floors', 'building_age', 'area_ratio',
       'time_index', 'log_area', 'has_parking', 'district_中正區', 'district_信義區',
       'district_內湖區', 'district_北投區', 'district_南港區', 'district_士林區',
       'district_大同區', 'district_大安區', 'district_文山區', 'district_松山區',
       'district_萬華區', 'building_type_公寓(5樓含以下無電梯)',
       'building_type_華廈(10層含以下有電梯)'],
      dtype='object')

In [28]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='rmse')
results_df

,model,rmse,mae,notes
1,Ridge,59318.687121,45335.042002,regularized linear
2,Lasso,59696.324643,45737.984241,regularized linear
0,Linear Regression (v2),59696.362770,45738.039962,baseline
3,Random Forest,81289.336763,62833.236005,tree-based
5,"XGBoost {'n_estimators': 100, 'max_depth': 4, ...",90400.824067,68894.429688,tuned
4,XGBoost,97852.745531,71994.000000,tree-based
7,"XGBoost {'n_estimators': 300, 'max_depth': 8, ...",100554.652523,73420.867188,tuned
6,"XGBoost {'n_estimators': 200, 'max_depth': 6, ...",103178.312140,75253.593750,tuned


# Notes and Analysis

## Model Comparison Insight — Why Ridge Outperformed Tree-Based Models

Ridge Regression performed best (lowest RMSE), while Random Forest and XGBoost significantly underperformed. This result suggests that the underlying structure of the dataset is largely linear rather than highly non-linear.

### Key Reasons

- **Strong linear signal in the data**
  - Features such as `area`, `building_age`, `district`, and `building_type` act as mostly additive contributors to price.
  - The problem closely resembles a classic *hedonic pricing model*, which is well-suited for linear regression.

- **Feature engineering already reduced non-linearity**
  - Transformations like `log_area`, `area_ratio`, and `time_index` have already linearized many relationships.
  - One-hot encoding of `district` and `building_type` turns categorical effects into simple additive shifts.

- **Limited feature interactions**
  - Tree-based models rely on strong interaction effects (e.g., conditional relationships between features), which are weak or absent in this dataset.

- **Target variable is already normalized**
  - Using `final_price_per_sqm` removes much of the scale-driven complexity, further simplifying relationships.

- **Noise and dataset size favor simpler models**
  - Real estate data contains inherent noise (pricing inconsistencies, parking effects, etc.).
  - In such settings, lower-variance models like Ridge generalize better than flexible tree-based models.

### Key Takeaway

The feature engineering process has effectively transformed the problem into a mostly **linear additive system**, which explains why Ridge Regression outperforms more complex tree-based models. Tree models are not failing — they simply have limited non-linear structure to exploit in the current dataset.